[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/42_label_smoothing_solution.ipynb)

# 🟢 Solution: Label Smoothing Loss

Reference solution for label smoothing loss.

$$y_{\text{smooth},k} = \begin{cases} 1 - \varepsilon & k = \text{correct class} \\ \varepsilon / (C - 1) & \text{otherwise} \end{cases}$$

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def label_smoothing(logits, targets, smoothing=0.1):
    N, C = logits.shape
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    soft_targets = torch.full_like(log_probs, smoothing / (C - 1))
    soft_targets.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing)
    return -(soft_targets * log_probs).sum(dim=-1).mean()

In [ ]:
torch.manual_seed(0)
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))

loss_smoothed = label_smoothing(logits, targets, smoothing=0.1)
loss_ce = torch.nn.functional.cross_entropy(logits, targets)

print(f"Label smoothing loss (eps=0.1): {loss_smoothed.item():.4f}")
print(f"Standard CE loss (eps=0.0):    {loss_ce.item():.4f}")

# With smoothing=0, should match standard CE
loss_no_smooth = label_smoothing(logits, targets, smoothing=0.0)
print(f"Label smoothing loss (eps=0.0): {loss_no_smooth.item():.4f}  (matches CE: {torch.allclose(loss_no_smooth, loss_ce)})") 

In [ ]:
from torch_judge import check
check("label_smoothing")